<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/Comms_3_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import datetime
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Chat history container
chat_history = []

# Custom CSS injection for improved styling
custom_css = """
<style>
    .chat-container {
        border: 1px solid #dcdcdc;
        border-radius: 8px;
        padding: 15px;
        background-color: #f9f9f9;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        max-width: 600px;
    }
    .chat-history-box {
        background-color: #ffffff;
        border: 1px solid #e0e0e0 !important;
        border-radius: 6px;
        font-family: 'Courier New', Courier, monospace;
        padding: 10px !important;
    }
    .chat-title {
        color: #2c3e50;
        font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
        margin-top: 0px;
        margin-bottom: 15px;
        border-bottom: 2px solid #3498db;
        padding-bottom: 5px;
    }
</style>
"""

# UI Elements
css_style = widgets.HTML(custom_css)
title_label = widgets.HTML("<h2 class='chat-title'>Simple Communications Client</h2>")
history_area = widgets.Output(layout={'border': 'none', 'height': '220px', 'overflow_y': 'auto'})

# Wrap history area inside an HTML box for applying custom CSS class
history_wrapper = widgets.VBox([history_area])
history_wrapper.add_class("chat-history-box")

username_input = widgets.Text(value='User', placeholder='Username', layout=widgets.Layout(width='25%'))
message_input = widgets.Text(placeholder='Type your message here...', layout=widgets.Layout(width='45%'))
send_button = widgets.Button(description='Send', button_style='primary', layout=widgets.Layout(width='15%'))
clear_button = widgets.Button(description='Clear', button_style='warning', layout=widgets.Layout(width='15%'))

def update_history():
    with history_area:
        history_area.clear_output()
        for timestamp, sender, msg in chat_history:
            if sender != "Client-Bot":
                print(f"[{timestamp}] {sender}: {msg}")
            else:
                print(f"[{timestamp}] Bot: {msg}")
        # Auto-scroll JavaScript execution
        display(HTML("<script>var el = document.querySelector('.chat-history-box'); if (el) { el.scrollTop = el.scrollHeight; }</script>"))

def on_send_clicked(b):
    global chat_history
    msg_text = message_input.value.strip()
    username = username_input.value.strip() or 'User'
    if msg_text:
        user_timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        chat_history.append((user_timestamp, username, msg_text))
        message_input.value = ''

        bot_timestamp = datetime.datetime.now().strftime("%H:%M:%S")
        chat_history.append((bot_timestamp, "Client-Bot", f"Received: '{msg_text}'"))

        # Keep only the last 10 messages
        chat_history = chat_history[-10:]
        update_history()

def on_clear_clicked(b):
    global df
    # Persist chat history to a local JSON file before clearing
    if chat_history:
        try:
            with open('chat_history_backup.json', 'w') as f:
                json.dump(chat_history, f, indent=4)
            print("Chat history successfully backed up to chat_history_backup.json")

            # Load into DataFrame globally
            df = pd.read_json('chat_history_backup.json')
            if not df.empty and len(df.columns) == 3:
                df.columns = ['Timestamp', 'Sender', 'Message']

            # Auto-plot message frequency
            with history_area:
                clear_output(wait=True)
                print("Generating Message Frequency visualization...")
                plt.figure(figsize=(6, 3.5))
                sns.countplot(data=df, x='Sender', palette='viridis')
                plt.title('Message Frequency by Sender')
                plt.xlabel('Sender')
                plt.ylabel('Number of Messages')
                plt.grid(axis='y', linestyle='--', alpha=0.7)
                plt.tight_layout()
                plt.show()

        except Exception as e:
            print(f"Failed to back up or plot chat history: {e}")
    chat_history.clear()
    if not 'df' in globals():
        update_history()

send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)
message_input.on_submit(on_send_clicked)

# Layout Arrangement
input_box = widgets.HBox([username_input, message_input, send_button, clear_button], layout=widgets.Layout(margin='10px 0px 0px 0px'))
ui_container = widgets.VBox([css_style, title_label, history_wrapper, input_box])
ui_container.add_class("chat-container")

display(ui_container)
update_history()

In [16]:
import pandas as pd
import os

file_path = 'chat_history_backup.json'

if os.path.exists(file_path):
    # Load the JSON data
    df = pd.read_json(file_path)
    # Add readable column names since the JSON stores it as a list of lists/tuples
    if not df.empty and len(df.columns) == 3:
        df.columns = ['Timestamp', 'Sender', 'Message']
    display(df)
else:
    print(f"The file '{file_path}' does not exist yet. Please send some messages and click 'Clear' in the chat client above to generate the file first!")

The file 'chat_history_backup.json' does not exist yet. Please send some messages and click 'Clear' in the chat client above to generate the file first!


In [17]:
def export_to_csv(dataframe, output_filename='chat_history_export.csv'):
    """
    Exports the given DataFrame to a CSV file.
    """
    if dataframe is None or dataframe.empty:
        print("DataFrame is empty or None. Please ensure you have chat history to export.")
        return
    try:
        dataframe.to_csv(output_filename, index=False)
        print(f"Chat history successfully exported to '{output_filename}'!")
    except Exception as e:
        print(f"An error occurred while exporting to CSV: {e}")

# Example usage (commented out until you have a populated 'df'):
# export_to_csv(df)

In [18]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_message_frequency(dataframe):
    if dataframe is None or dataframe.empty:
        print("The DataFrame is empty. Please make sure you have generated the backup and loaded it in the cells above!")
        return

    plt.figure(figsize=(8, 5))
    sns.countplot(data=dataframe, x='Sender', palette='viridis')
    plt.title('Message Frequency by Sender')
    plt.xlabel('Sender')
    plt.ylabel('Number of Messages')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Safely call the visualization function with the loaded data if it exists
if 'df' in globals():
    plot_message_frequency(df)
else:
    print("The variable 'df' is not defined yet. Please make sure to send some messages, click 'Clear' in the chat interface, and run Cell 4a5e4ef5 to load the data first.")

The variable 'df' is not defined yet. Please make sure to send some messages, click 'Clear' in the chat interface, and run Cell 4a5e4ef5 to load the data first.
